In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- 1. SETUP ---
MAIN_DATA_FILE = 'C:/Users/511232/Desktop/DSS/MERGING GOOGLESHEETS QUESTIONNAIRES/codes/arabic_questionnaires.xlsx'
CRITERIA_FILE = 'C:/Users/511232/Desktop/criterias.xlsx'

# Load data
main_df = pd.read_excel(MAIN_DATA_FILE)
criteria_df = pd.read_excel(CRITERIA_FILE).dropna(subset=['Indicator_Ar'])[['Indicator_Ar', 'criteria']]

# Force numeric values (Invalid text becomes NaN)
main_df['val_numeric'] = pd.to_numeric(main_df['القيمة'], errors='coerce')

# Bracket Definitions
conds = [
    (main_df['السنة'] >= 2010) & (main_df['السنة'] <= 2014),
    (main_df['السنة'] >= 2015) & (main_df['السنة'] <= 2019),
    (main_df['السنة'] >= 2020) & (main_df['السنة'] <= 2026)
]
bins = ['2010-2014', '2015-2019', '2020-2026']

In [ ]:
# --- 2. GENERAL AVAILABILITY ---
# A. Collapse to unique years per indicator/country/bracket
df_gen = main_df.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
df_gen['bracket'] = np.select(conds, bins, default='Other')
df_gen = df_gen[df_gen['bracket'] != 'Other']
df_gen['is_valid'] = df_gen['val_numeric'].notna().astype(int)

# B. Sum valid years per bracket
gen_brk = df_gen.groupby(['المؤشر', 'الدولة', 'bracket'])['is_valid'].sum().reset_index()

# C. Calculate Group Size (How many brackets exist for each pair)
gen_brk['group_size'] = gen_brk.groupby(['المؤشر', 'الدولة'])['bracket'].transform('size')

# D. Merge Criteria and Apply Logic
gen_final = pd.merge(gen_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
# Availability = 1 if (group_size is 3) AND (every bracket count >= criteria)
gen_final['is_ok'] = ((gen_final['group_size'] == 3) & (gen_final['is_valid'] >= gen_final['criteria'])).astype(int)

# E. Final Collapse to 0/1 per Country/Indicator
final_gen_report = gen_final.groupby(['المؤشر', 'الدولة'])['is_ok'].min().reset_index(name='general_availability')
final_gen_report.to_excel('general_availability.xlsx', index=False)

In [ ]:
# --- 3. NATIONALITY AVAILABILITY ---
# A. Initial Clean
df_nat = main_df.dropna(subset=['المواطنة', 'val_numeric'])
df_nat = df_nat[~df_nat['المواطنة'].isin(['Not applicable', 'غير متاح', 'Total'])]

# B. Collapse to unique years (Checking if EITHER has data)
df_nat_yearly = df_nat.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
df_nat_yearly['bracket'] = np.select(conds, bins, default='Other')
df_nat_yearly = df_nat_yearly[df_nat_yearly['bracket'] != 'Other']

# C. Count years in bracket and get Group Size
nat_brk = df_nat_yearly.groupby(['المؤشر', 'الدولة', 'bracket']).size().reset_index(name='year_count')
nat_brk['group_size'] = nat_brk.groupby(['المؤشر', 'الدولة'])['bracket'].transform('size')

# D. Merge Criteria and Apply Logic
nat_final = pd.merge(nat_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
nat_final['is_ok'] = ((nat_final['group_size'] == 3) & (nat_final['year_count'] >= nat_final['criteria'])).astype(int)

# E. Final Collapse
final_nat_report = nat_final.groupby(['المؤشر', 'الدولة'])['is_ok'].min().reset_index(name='nationality_availability')
final_nat_report.to_excel('nationality_availability.xlsx', index=False)

In [ ]:
# --- 4. SEX AVAILABILITY ---
# Remove blanks
df_sex = main_df.dropna(subset=['الجنس', 'val_numeric'])
df_sex = df_sex[~df_sex['الجنس'].isin(['Not applicable', 'غير متاح', 'Total'])]

# Collapse to Year
sex_yearly = df_sex.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
sex_yearly['bracket'] = np.select(conds, bins, default='Other')

# Sum years in bracket
sex_brk = sex_yearly[sex_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket']).size().reset_index(name='count')
sex_merged = pd.merge(sex_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
sex_merged['is_met'] = (sex_merged['count'] >= sex_merged['criteria']).astype(int)

# Final Check
final_sex = sex_merged.groupby(['المؤشر', 'الدولة']).filter(lambda x: len(x) == 3 and x['is_met'].all())
final_sex = final_sex[['المؤشر', 'الدولة']].drop_duplicates()
final_sex['sex_availability'] = 1

all_pairs_sex = sex_merged[['المؤشر', 'الدولة']].drop_duplicates()
final_sex = pd.merge(all_pairs_sex, final_sex, on=['المؤشر', 'الدولة'], how='left').fillna(0)
final_sex.to_excel('sex_availability.xlsx', index=False)

print("Process Complete. 3 files saved.")